# Блокнот 2.4–2.5. pandas на настоящих данных

Курс «Машинное обучение с нуля», уроки 2.4 и 2.5. Данные — Palmer Penguins: 344 пингвина трёх видов с островов Антарктиды (Gorman и соавт., 2014). seaborn скачивает таблицу из интернета, поэтому в Colab нужен доступ к сети.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

df = sns.load_dataset("penguins")
df.head()

## 1. Первый взгляд на таблицу

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
df["species"].value_counts()

**Задание 1.** Сколько пропусков в каждом столбце? Сохраните результат `df.isna().sum()` в `missing`.

In [ ]:
missing = ...  # ваш код
missing

In [ ]:
assert missing["sex"] == 11 and missing["body_mass_g"] == 2
print("Верно!")

## 2. Фильтрация

Условие на столбец даёт **маску** — Series из True и False. Таблицу фильтруют маской: `df[маска]`. Несколько условий соединяют через `&` (и), `|` (или), `~` (не), и каждое условие берут в скобки.

In [ ]:
heavy = df[df["body_mass_g"] > 5000]
print(len(heavy))
heavy["species"].value_counts()

**Задание 2.** Сколько пингвинов вида Gentoo весят больше 5000 г? Сохраните число в `n_heavy_gentoo`.

In [ ]:
n_heavy_gentoo = ...  # ваш код

In [ ]:
assert n_heavy_gentoo == 61, f"получилось {n_heavy_gentoo}"
print("Верно!")

## 3. Пропуски

Строки с пропусками можно удалить (`dropna`) или заполнить (`fillna`). Здесь мы заполняем медианой всей таблицы — на настоящей задаче медиану берут **только по обучающей выборке** (урок 1.4).

In [ ]:
filled = df.copy()
filled["body_mass_g"] = filled["body_mass_g"].fillna(filled["body_mass_g"].median())
print(filled["body_mass_g"].isna().sum())
clean = df.dropna()
print(len(df), "→", len(clean))

## 4. Группировка

**Задание 3.** Найдите среднюю массу пингвинов каждого вида (`groupby` + `mean`) и сохраните в `mass_by_species`.

In [ ]:
mass_by_species = ...  # ваш код
mass_by_species.round(1)

In [ ]:
assert round(mass_by_species["Gentoo"]) == 5076, f"получилось {mass_by_species}"
print("Верно!")

In [ ]:
df.groupby(["species", "sex"])["flipper_length_mm"].agg(["mean", "count"]).round(1)

## 5. Объединение таблиц

У нас есть справочник островов. Присоединим его к пингвинам по столбцу `island`.

In [ ]:
islands = pd.DataFrame({
    "island": ["Biscoe", "Dream", "Torgersen"],
    "station": ["Палмер", "Палмер", "Палмер"],
    "area_km2": [12.0, 3.2, 0.1],
})
merged = df.merge(islands, on="island", how="left")
merged[["species", "island", "area_km2"]].head()

## 6. Признаки для модели

Модели нужны числа. Категории превращаем в столбцы из 0 и 1 (one-hot), числа — масштабируем.

**Задание 4.** Сделайте one-hot кодирование столбца `island` функцией `pd.get_dummies` и сохраните в `island_cols`. Должно получиться три столбца.

In [ ]:
island_cols = ...  # ваш код
island_cols.head()

In [ ]:
assert list(island_cols.columns) == ["island_Biscoe", "island_Dream", "island_Torgersen"]
assert island_cols.sum(axis=1).eq(1).all(), "в каждой строке должна быть ровно одна единица"
print("Верно!")

**Задание 5.** Добавьте признак `bill_ratio` — отношение длины клюва к его глубине (`bill_length_mm / bill_depth_mm`).

In [ ]:
df["bill_ratio"] = ...  # ваш код
df.groupby("species")["bill_ratio"].mean().round(2)

In [ ]:
assert "bill_ratio" in df and abs(df.loc[0, "bill_ratio"] - 39.1 / 18.7) < 1e-9
print("Верно! У Gentoo клюв заметно длиннее и тоньше.")